In [1]:

from datasets import load_dataset
ds = load_dataset("leduckhai/VietMed-NER")
ds

d:\project\clinic_management_system_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['words', 'tags', 'labels', 'text', 'audio', 'duration'],
        num_rows: 4616
    })
    validation: Dataset({
        features: ['words', 'tags', 'labels', 'text', 'audio', 'duration'],
        num_rows: 1154
    })
    test: Dataset({
        features: ['words', 'tags', 'labels', 'text', 'audio', 'duration'],
        num_rows: 3497
    })
})

In [2]:
ds = ds.remove_columns(["audio", "duration"])

print(ds)

DatasetDict({
    train: Dataset({
        features: ['words', 'tags', 'labels', 'text'],
        num_rows: 4616
    })
    validation: Dataset({
        features: ['words', 'tags', 'labels', 'text'],
        num_rows: 1154
    })
    test: Dataset({
        features: ['words', 'tags', 'labels', 'text'],
        num_rows: 3497
    })
})


In [ ]:
ds.save_to_disk("./../dataset/vietmed_ner")

Saving the dataset (1/1 shards): 100%|██████████| 3497/3497 [00:00<00:00, 902780.89 examples/s]


In [6]:
for split in ds:
    ds[split].to_json(
        f"./dataset/vietmed_ner/{split}.json",
        orient="records",
        force_ascii=False
    )

Creating json from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 104.53ba/s]


In [ ]:
import pandas as pd

df = pd.read_csv(
    "training/dataset/disease/disease_knowledge_base.csv"
)

print(df.columns)
print(df.shape)
print(df.head(20))
print(df["disease"].nunique())

In [ ]:
from datasets import Dataset
ds = Dataset.from_file(
    "./../dataset/vietmed_ner/train/data-00000-of-00001.arrow"
)


print(ds)
print(ds.column_names)
print(ds[0]["text"])

Dataset({
    features: ['words', 'tags', 'labels', 'text'],
    num_rows: 4616
})
['words', 'tags', 'labels', 'text']
{'words': ['thì', 'cũng', 'giống', 'như', 'ba', 'má', 'mình', 'đã', 'từng', 'bị', 'béo', 'phì', 'rồi', 'bị', 'bệnh', 'này', 'bệnh', 'kia', 'những', 'người', 'thân', 'quen', 'của', 'mình', 'mình', 'biết', 'bị'], 'tags': [22, 22, 22, 22, 13, 13, 22, 22, 22, 22, 2, 8, 22, 2, 8, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22], 'labels': ['0', '0', '0', '0', 'B-GENDER', 'B-GENDER', '0', '0', '0', '0', 'B-DISEASESYMTOM', 'I-DISEASESYMTOM', '0', 'B-DISEASESYMTOM', 'I-DISEASESYMTOM', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0'], 'text': 'thì cũng giống như ba má mình đã từng bị béo phì rồi bị bệnh này bệnh kia những người thân quen của mình mình biết bị'}


In [ ]:
filtered = ds["train"].filter(
    lambda x: "mắt" in x["text"].lower()
)

print("Số mẫu:", len(filtered))

for row in filtered:
    print("=" * 80)
    print("TEXT:", row["text"])
    print("WORDS:", row["words"])
    print("TAGS:", row["tags"])

Filter: 100%|██████████| 4616/4616 [00:00<00:00, 210848.01 examples/s]

Số mẫu: 34
TEXT: cái bệnh lý như là các cái thiểu năng vành này rồi là những cái cái tình trạng đau đầu hoa mắt chóng mặt của các cái
WORDS: ['cái', 'bệnh', 'lý', 'như', 'là', 'các', 'cái', 'thiểu', 'năng', 'vành', 'này', 'rồi', 'là', 'những', 'cái', 'cái', 'tình', 'trạng', 'đau', 'đầu', 'hoa', 'mắt', 'chóng', 'mặt', 'của', 'các', 'cái']
TAGS: [22, 22, 22, 22, 22, 22, 22, 2, 8, 8, 22, 22, 22, 22, 22, 22, 22, 22, 2, 8, 2, 8, 2, 8, 22, 22, 22]
TEXT: giúp cho máu không lên não lên tim tốt này giúp cho cải thiện các triệu chứng của bác đang có đấy ạ đau đầu hoa mắt
WORDS: ['giúp', 'cho', 'máu', 'không', 'lên', 'não', 'lên', 'tim', 'tốt', 'này', 'giúp', 'cho', 'cải', 'thiện', 'các', 'triệu', 'chứng', 'của', 'bác', 'đang', 'có', 'đấy', 'ạ', 'đau', 'đầu', 'hoa', 'mắt']
TAGS: [22, 22, 34, 22, 22, 34, 22, 34, 22, 22, 22, 22, 17, 20, 22, 2, 8, 22, 14, 22, 22, 22, 22, 2, 8, 2, 8]
TEXT: liên quan đến chủ đề ngày hôm nay những ai mà chúng ta đang gặp phải cái tình trạng là đau đầu hoa mắt chóng mặt

In [15]:
keywords = [
    "đau mắt",
    "đỏ mắt",
    "ngứa mắt",
    "rát mắt",
    "cộm mắt",
    "sưng mắt",
    "mờ mắt",
    "chảy nước mắt",
    "hoa mắt",
]

for keyword in keywords:
    filtered = ds["train"].filter(
        lambda x, keyword=keyword:
            keyword in x["text"].lower()
    )

    print(
        f"{keyword:20} -> {len(filtered)} samples"
    )

Filter: 100%|██████████| 4616/4616 [00:00<00:00, 222281.11 examples/s]


đau mắt              -> 0 samples


Filter: 100%|██████████| 4616/4616 [00:00<00:00, 227684.31 examples/s]


đỏ mắt               -> 0 samples


Filter: 100%|██████████| 4616/4616 [00:00<00:00, 232019.98 examples/s]


...

mờ mắt               -> 0 samples


Filter: 100%|██████████| 4616/4616 [00:00<00:00, 231562.10 examples/s]


chảy nước mắt        -> 0 samples


Filter: 100%|██████████| 4616/4616 [00:00<00:00, 213842.89 examples/s]

hoa mắt              -> 24 samples


In [17]:
print(ds["train"].features["tags"])
print(ds["train"].features["labels"])

List(Value('int64'))
List(Value('string'))
